In [1]:
# !cp -r /content/drive/MyDrive/Data.zip /content/
# !unzip /content/Data.zip -d /content

Streaming output truncated to the last 5000 lines.
  inflating: /content/Data/Val/Pepper_Healthy/0e5d9ac1-4de8-491e-ab95-809850866805___JR_HL 8534_new30degFlipLR.JPG  
  inflating: /content/Data/Val/Pepper_Healthy/0e69c47d-72c6-4fc6-9437-910c95b183dc___JR_HL 8113_new30degFlipLR.JPG  
  inflating: /content/Data/Val/Pepper_Healthy/0eabc3c2-d492-4227-90d8-14dab9fd4a9a___JR_HL 8699.JPG  
  inflating: /content/Data/Val/Pepper_Healthy/0eabc3c2-d492-4227-90d8-14dab9fd4a9a___JR_HL 8699_newPixel25.JPG  
  inflating: /content/Data/Val/Pepper_Healthy/0eb476b9-8b21-4b24-adef-7db813abbca3___JR_HL 7926.JPG  
  inflating: /content/Data/Val/Pepper_Healthy/0eb476b9-8b21-4b24-adef-7db813abbca3___JR_HL 7926_newPixel25.JPG  
  inflating: /content/Data/Val/Pepper_Healthy/0eeb924f-88db-44e6-a278-e016fa2d25a4___JR_HL 8016_flipTB.JPG  
  inflating: /content/Data/Val/Pepper_Healthy/0f56ffda-5d57-4a61-8f7f-37f22a02520a___JR_HL 8589_newPixel25.JPG  
  inflating: /content/Data/Val/Pepper_Healthy/0fbb1d8d-63ff-406

In [12]:
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights
import torch.optim as optim

In [13]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

In [14]:
trainData = ImageFolder("/content/Data/Train",transform=train_transform)
validationData = ImageFolder("/content/Data/Val",transform=val_transform)

trainDataLoader = DataLoader(
    trainData,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
    )
validationDataLoader = DataLoader(validationData,batch_size=32,shuffle=False,num_workers=2,pin_memory=True,
    persistent_workers=True)

In [15]:
model = resnet50(weights=ResNet50_Weights.DEFAULT)

In [16]:


# Freeze everything
for param in model.parameters():
    param.requires_grad = False

# Replace classifier
model.fc = nn.Linear(model.fc.in_features, 29)

# Unfreeze last residual block group
for param in model.layer4.parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

In [17]:
epochs = 10
best_val_loss = float('inf')
for epoch in range(epochs):
  model.train()
  epoch_training_loss=0.0
  correct_train=0
  total_train=0
  for image,label in trainDataLoader:
    image , label = image.to(device),label.to(device)
    optimizer.zero_grad()
    output = model(image)
    loss = criterion(output,label)
    loss.backward()
    optimizer.step()
    epoch_training_loss += loss.item()
    _,predicted = torch.max(output,1)
    total_train += label.size(0)
    correct_train += (predicted == label).sum().item()
  avg_train_loss = epoch_training_loss / len(trainDataLoader)
  train_acc = 100 * correct_train / total_train
  model.eval()
  running_val_loss =0.0
  correct_val=0
  total_val=0
  with torch.no_grad():
    for image,label in validationDataLoader:
      image, label = image.to(device), label.to(device)
      output = model(image)
      loss=criterion(output,label)
      running_val_loss += loss.item()
      _,predicted = torch.max(output,1)
      total_val += label.size(0)
      correct_val += (predicted == label).sum().item()
  avg_val_loss = running_val_loss / len(validationDataLoader)
  val_acc = 100 * correct_val / total_val

  print(f'Epoch {epoch+1}/{epochs}, Training Loss: {avg_train_loss:.4f}, Training Accuracy: {train_acc:.2f}%, Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {val_acc:.2f}%')
  if avg_val_loss < best_val_loss:
    best_val_loss = avg_val_loss
    torch.save(model.state_dict(), 'best_model_resNet50_Finetuned.pt')


Epoch 1/10, Training Loss: 0.4908, Training Accuracy: 87.02%, Validation Loss: 0.0896, Validation Accuracy: 97.44%
Epoch 2/10, Training Loss: 0.1493, Training Accuracy: 95.37%, Validation Loss: 0.0565, Validation Accuracy: 98.34%
Epoch 3/10, Training Loss: 0.1113, Training Accuracy: 96.50%, Validation Loss: 0.0515, Validation Accuracy: 98.33%
Epoch 4/10, Training Loss: 0.0931, Training Accuracy: 96.98%, Validation Loss: 0.0423, Validation Accuracy: 98.57%
Epoch 5/10, Training Loss: 0.0817, Training Accuracy: 97.36%, Validation Loss: 0.0316, Validation Accuracy: 98.96%
Epoch 6/10, Training Loss: 0.0758, Training Accuracy: 97.58%, Validation Loss: 0.0311, Validation Accuracy: 99.06%
Epoch 7/10, Training Loss: 0.0687, Training Accuracy: 97.72%, Validation Loss: 0.0249, Validation Accuracy: 99.23%
Epoch 8/10, Training Loss: 0.0630, Training Accuracy: 97.97%, Validation Loss: 0.0238, Validation Accuracy: 99.22%
Epoch 9/10, Training Loss: 0.0602, Training Accuracy: 98.08%, Validation Loss: 0

In [19]:
model.load_state_dict(torch.load("/content/best_model_resNet50_Finetuned.pt"))

<All keys matched successfully>

In [20]:
testData = ImageFolder("/content/Data/Test",transform=val_transform)
testDataLoader = DataLoader(
    testData,
    batch_size=32,
    num_workers=2,
    shuffle=False,
    pin_memory=True,
    persistent_workers=True
)

In [21]:
model.to(device)
model.eval()
all_labes=[]
all_preds=[]
with torch.no_grad():
  for images, labels in testDataLoader:
    images=images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    all_labes.extend(labels.cpu().numpy())

    all_preds.extend(predicted.cpu().numpy())

In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
accuracy = accuracy_score(all_labes, all_preds)
precision = precision_score(all_labes, all_preds, average='weighted')
recall = recall_score(all_labes, all_preds, average='weighted')
f1 = f1_score(all_labes, all_preds, average='weighted')
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

Accuracy: 0.9971
Precision: 0.9971
Recall: 0.9971
F1 Score: 0.9971
